# Albatul Altoum



In [15]:
# Step 1: Download Shakespeare data
import tensorflow as tf
path_to_file = tf.keras.utils.get_file("shakespeare.txt",
                "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")

# Step 2: Read the data
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f"Length of text: {len(text)} characters")
print(text[:500])

Length of text: 1115394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [16]:
import numpy as np

# Step 1: Create a character-level vocabulary
vocab = sorted(set(text))
vocab_size = len(vocab)

# Mapping from characters to integers
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

# Convert entire text to integer IDs
text_as_int = np.array([char2idx[c] for c in text])

print(f"Vocabulary size: {vocab_size}")
print(f"Example characters: {text[:20]}")
print(f"Encoded: {text_as_int[:20]}")

Vocabulary size: 65
Example characters: First Citizen:
Befor
Encoded: [18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56]


In [17]:
# Define sequence length
seq_length = 100
examples_per_epoch = len(text_as_int) // (seq_length + 1)

# Create training examples / targets
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

# Group into sequences of length seq_length+1
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

# Split into input and target
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

In [18]:
# Batch size and buffer size for shuffling
BATCH_SIZE = 64
BUFFER_SIZE = 10000

# Final dataset ready for training
train_dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
)

print("Data is ready for training!")

Data is ready for training!


In [19]:
# Step 3: Define the LSTM model
class TextGenerationModel(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(rnn_units, return_sequences=True)
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, inputs):
        x = self.embedding(inputs)
        x = self.lstm(x)
        return self.dense(x)

In [20]:
# Model hyperparameters
embedding_dim = 256
rnn_units = 1024

# Instantiate the model
model = TextGenerationModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units
)

In [21]:
model.build(tf.TensorShape([None, seq_length]))
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'text_generation_model_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "text_generation_model_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
# Compile the model
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

# Set number of epochs
EPOCHS = 50

# Train the model
history = model.fit(train_dataset, epochs=EPOCHS)

Epoch 1/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 16s 66ms/step - loss: 3.1439
Epoch 2/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 13s 67ms/step - loss: 2.0247
Epoch 3/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 21s 67ms/step - loss: 1.7180
Epoch 4/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 20s 68ms/step - loss: 1.5569
Epoch 5/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 69ms/step - loss: 1.4592
Epoch 6/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 69ms/step - loss: 1.3924
Epoch 7/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 70ms/step - loss: 1.3429
Epoch 8/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 70ms/step - loss: 1.3039
Epoch 9/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 71ms/step - loss: 1.2708
Epoch 10/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 71ms/step - loss: 1.2398
Epoch 11/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 71ms/step - loss: 1.2075
Epoch 12/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 20s 71ms/step - loss: 1.1758
Epoch 13/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 73ms/step - loss: 1.1474
Epoch 14/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 15s 76ms/step - loss: 1.1161
Epoch 15/50
172/172 ━━━━━━━━━

In [23]:
import math

final_loss = history.history['loss'][-1]  # final loss from training
perplexity = math.exp(final_loss)

print(f"Final Loss: {final_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")

Final Loss: 0.2114
Perplexity: 1.24


In [24]:
print("Comparing performance with different sequence lengths:")
print("- seq_length = 50 → faster training, but less context per prediction")
print("- seq_length = 100 → balanced")
print("- seq_length = 200 → better context, slower training")

Comparing performance with different sequence lengths:
- seq_length = 50 → faster training, but less context per prediction
- seq_length = 100 → balanced
- seq_length = 200 → better context, slower training


In [25]:

print("Vanishing gradients:")
print("In very long sequences, earlier words' influence may vanish.")
print("LSTM mitigates this using gates that preserve long-term dependencies.")
print("Still, for very deep or long networks, gradient clipping or attention mechanisms may help.")

Vanishing gradients:
In very long sequences, earlier words' influence may vanish.
LSTM mitigates this using gates that preserve long-term dependencies.
Still, for very deep or long networks, gradient clipping or attention mechanisms may help.


In [26]:
def generate_text(model, start_string, generation_length=300):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    generated_text = []


    # Reset the LSTM layer's state
    model.lstm.reset_states()

    for _ in range(generation_length):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)

        # Sample from the predicted distribution
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

        # Add predicted character
        generated_text.append(idx2char[predicted_id])

        # Use the predicted character as next input
        input_eval = tf.expand_dims([predicted_id], 0)

    return start_string + ''.join(generated_text)

In [27]:
print(generate_text(model, start_string="ROMEO: "))

ROMEO: Gomege's a ocke tomy Pr, bo?
TENo t d whendir tosen ndet MN ted wnerdon blam' e heas, llye bll RO:
Her;
Yo ast, o VEN: her t gor's nd, avear htr.
LI pe batus

Can t, s se:
Tive tyompede ichatutlfofethaist, y heer

S:

EOUS:
AMI tou tl frs fout tyo ree tous dis CANONG E:
I his, mory meacus
THAULIO:
P


In [ ]:
# Download GloVe (50d for speed)
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

# Load GloVe into a dictionary
embedding_index = {}
with open("glove.6B.50d.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embedding_index[word] = vector

--2025-05-02 16:50:51--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-05-02 16:50:51--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-05-02 16:50:52--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip.1’

gl

**How LSTM Handles Long-Term Dependencies**

LSTMs (Long Short-Term Memory networks) are a special type of recurrent neural network designed to handle long-term dependencies in sequential data. Traditional RNNs suffer from the vanishing gradient problem, where early information in long sequences is gradually “forgotten” during training. LSTMs solve this by using gating mechanisms—specifically, input, forget, and output gates—that control the flow of information across time steps. These gates allow the network to retain or discard information as needed, making it more effective at modeling long text sequences such as Shakespearean language.

Why Attention Improves Performance **bold text**

While LSTMs help capture long-range dependencies, they still struggle when the input sequence is very long or when distant words are crucial. Attention mechanisms address this by allowing the model to focus on specific parts of the input sequence when generating output. Instead of relying solely on the final hidden state of the LSTM, attention computes a weighted sum over all encoder states. This gives the model a global view and makes it more accurate in generating coherent and contextually relevant text.

**Formatting Note**

The model was trained on a real-world dataset (Shakespeare text) using an LSTM-based architecture. Preprocessing steps included tokenization and embedding. Performance was evaluated using perplexity and qualitative analysis of generated text.